In [17]:
import streamlit as st
from libigc import Flight
from pathlib import Path
from datetime import datetime
import pandas as pd

In [10]:
root_dir = "c:/Users/402824/repos/igc-flight-analysis"
data_dir = Path("data/raw")
data_dir = Path(root_dir) / data_dir
available_files = list(data_dir.rglob("*.igc"))
file_options = [str(file.relative_to(data_dir)) for file in available_files]
file_options[0]

'2025-06-18\\458F30_1750261236_1750261791.igc'

In [58]:
file_path = file_options[80]
full_file_path = data_dir / file_path
flight = Flight.create_from_file(full_file_path)
print(f"file_path: {file_path}")
print(f"full_file_path: {full_file_path}")
flight

file_path: 2025-06-30\EKSL\D02250_1751293960_1751297676.igc
full_file_path: c:\Users\402824\repos\igc-flight-analysis\data\raw\2025-06-30\EKSL\D02250_1751293960_1751297676.igc


In [61]:
# Query the igc_flight_analysis database. Use table: igc_files column file_path to find the id of the flight. Join that to flight_id from the flights table
import sqlite3

# Set working directory to the .. parent directory of the current script

conn = sqlite3.connect("..\igc_flight_analysis.db")

query = """
SELECT
    i.*,
    f.*
FROM flights f
JOIN igc_files i ON i.flight_id = f.id
WHERE i.file_path = ?
"""
cursor = conn.cursor()
cursor.execute(query, ("raw\\" + file_path,))
result = cursor.fetchone()

# To pandas DataFrame
columns = [column[0] for column in cursor.description]
df = pd.DataFrame([result], columns=columns)
df.head()

<>:6: SyntaxWarning: invalid escape sequence '\i'
<>:6: SyntaxWarning: invalid escape sequence '\i'
C:\Users\402824\AppData\Local\Temp\ipykernel_13144\4067283995.py:6: SyntaxWarning: invalid escape sequence '\i'
  conn = sqlite3.connect("..\igc_flight_analysis.db")


,id,flight_id,file_path,downloaded_at,id,device_address,report_id,start_tsp,stop_tsp,start_time,stop_time,duration_sec,max_alt,max_height,towing,tow_id,warn
0,973,973,raw\2025-06-30\EKSL\D02250_1751293960_17512976...,2025-06-30 20:56:08.378494,973,D02250,116,1751293960,1751297676,16h32,17h34,3716,906,872,0,None,0


In [ ]:
# Flight Class. Attributes. 
# Note: this should probably be done adhoc in the browser.
print(flight.takeoff_fix.timestamp)
print(flight.landing_fix.timestamp)
print(flight.thermals)
print(flight.glides)
print(flight.takeoff_fix)
print(flight.landing_fix)
print(flight.alt_source)
print(flight.valid)
print(flight.notes)


1751293960.0
1751297670.0
[Thermal(vertical_velocity=0.40 m/s, duration=17m 34s), Thermal(vertical_velocity=0.16 m/s, duration=3m 31s), Thermal(vertical_velocity=0.08 m/s, duration=3m 56s), Thermal(vertical_velocity=1.11 m/s, duration=4m 57s)]
[Glide(dist=12.85 km, avg_speed=101.22 kph, avg L/D=31.26 duration=7m 37s), Glide(dist=7.88 km, avg_speed=82.50 kph, avg L/D=-96.14 duration=5m 44s), Glide(dist=14.21 km, avg_speed=99.13 kph, avg L/D=-26.56 duration=8m 36s), Glide(dist=0.71 km, avg_speed=77.16 kph, avg L/D=-353.67 duration=0m 33s), Glide(dist=15.57 km, avg_speed=99.75 kph, avg L/D=-26.22 duration=9m 22s)]
GNSSFix(rawtime=14:32:40, lat=55.452533, lon=11.648683, press_alt=0.0, gnss_alt=32.0)
GNSSFix(rawtime=15:34:30, lat=55.452200, lon=11.647100, press_alt=0.0, gnss_alt=33.0)
GNSS
True
['Warning: average pressure altitude change between fixes is: 0.000000. It is lower than the minimum: 0.010000.']


In [ ]:
# Use the libigc thermal class to analyze the thermals.
# Note, could create a table with the thermal attributes linking it to the flight.
for thermal in flight.thermals:
    print(thermal.time_change())
    print(thermal.alt_change())
    print(thermal.vertical_velocity())
    print(thermal.enter_fix)
    print(thermal.exit_fix)

1054.0
421.0
0.39943074003795065
GNSSFix(rawtime=14:40:17, lat=55.457500, lon=11.632667, press_alt=0.0, gnss_alt=443.0)
GNSSFix(rawtime=14:57:51, lat=55.438583, lon=11.655117, press_alt=0.0, gnss_alt=864.0)
211.0
33.0
0.15639810426540285
GNSSFix(rawtime=15:03:35, lat=55.471533, lon=11.590333, press_alt=0.0, gnss_alt=782.0)
GNSSFix(rawtime=15:07:06, lat=55.469050, lon=11.586567, press_alt=0.0, gnss_alt=815.0)
236.0
18.0
0.07627118644067797
GNSSFix(rawtime=15:15:42, lat=55.439450, lon=11.633700, press_alt=0.0, gnss_alt=280.0)
GNSSFix(rawtime=15:19:38, lat=55.441933, lon=11.638550, press_alt=0.0, gnss_alt=298.0)
297.0
331.0
1.1144781144781144
GNSSFix(rawtime=15:20:11, lat=55.440650, lon=11.651100, press_alt=0.0, gnss_alt=296.0)
GNSSFix(rawtime=15:25:08, lat=55.433150, lon=11.655250, press_alt=0.0, gnss_alt=627.0)


In [72]:
# Use the libigc thermal class to analyze the thermals.
# Note, could create a table with the thermal attributes linking it to the flight.
for glide in flight.glides:
    print(glide.time_change())
    print(glide.speed())
    print(glide.alt_change())
    print(glide.glide_ratio())
    print(glide.enter_fix)
    print(glide.exit_fix)
    print(glide.track_length)

457.0
101.22225162292486
411.0
31.264239653742
GNSSFix(rawtime=14:32:40, lat=55.452533, lon=11.648683, press_alt=0.0, gnss_alt=32.0)
GNSSFix(rawtime=14:40:17, lat=55.457500, lon=11.632667, press_alt=0.0, gnss_alt=443.0)
12.849602497687961
344.0
82.50175887174962
-82.0
-96.14026101585999
GNSSFix(rawtime=14:57:51, lat=55.438583, lon=11.655117, press_alt=0.0, gnss_alt=864.0)
GNSSFix(rawtime=15:03:35, lat=55.471533, lon=11.590333, press_alt=0.0, gnss_alt=782.0)
7.883501403300519
516.0
99.12790743760593
-535.0
-26.55763252222464
GNSSFix(rawtime=15:07:06, lat=55.469050, lon=11.586567, press_alt=0.0, gnss_alt=815.0)
GNSSFix(rawtime=15:15:42, lat=55.439450, lon=11.633700, press_alt=0.0, gnss_alt=280.0)
14.208333399390183
33.0
77.16342123335295
-2.0
-353.6656806528677
GNSSFix(rawtime=15:19:38, lat=55.441933, lon=11.638550, press_alt=0.0, gnss_alt=298.0)
GNSSFix(rawtime=15:20:11, lat=55.440650, lon=11.651100, press_alt=0.0, gnss_alt=296.0)
0.7073313613057354
562.0
99.75064217952767
-594.0
-26.21

In [ ]:
# This script can be migrated to the ingestion script to create a table of track points.
track_points = [
    {
        "File Path": file_path, # path to igc file. This is a key to table igc_files table
        "Timestamp": datetime.fromtimestamp(fix.timestamp),  # Convert timestamp to datetime
        "Latitude": fix.lat, # Latitude
        "Longitude": fix.lon, # Longitude
        "Altitude": fix.alt, # Altitude
        "Ground Speed": fix.gsp, # Ground speed
        "Bearing": fix.bearing, # Bearing
        "Bearing Change Rate": fix.bearing_change_rate, # Bearing change rate
        "Flying": fix.flying, # Boolean if the aircraft is flying
        "Circling": fix.circling, # Boolean if the aircraft is circling
    }
    for fix in flight.fixes
]
track_points_df = pd.DataFrame(track_points)
track_points_df.head()

AttributeError: 'GNSSFix' object has no attribute 'vertical_velocity'